# Teachable machine
By now, you have already tried out Google’s Teachable Machine. In this notebook, we’ll delve deeper into how this Teachable Machine works. You will learn how to adapt an existing AI model so that it works for the data you have. This technique is usually called **transfer learning**.
## Transfer learning
In transfer learning, we start from an existing AI model. We adapt this model for our task. Here, we use the ImageNet model as a base. This model was trained with more than a million images of 1,000 different objects. It is therefore already very good at detecting these objects. In this notebook you add a layer to ImageNet and train that layer to recognize paper and PMD.

Before we can start, we import the necessary libraries. The TensorFlow, Keras, and Sklearn libraries all contain various functions that make it easier to work with AI models. TensorFlow and Keras focus specifically on neural networks. Sklearn is a more general library with which you can also build other types of AI models. In addition, we also load the NumPy library. This makes it easier to work with matrices.
Furthermore, at Dwengo we have also written a number of functions that make it easier to read in your data and display it in the notebook. We load these functions via the **helpers** module. If you want to take a closer look at the code, you can open the file *scripts/helpers.py*.

In [ ]:
from scripts import helpers

import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet import preprocess_input, decode_predictions
import numpy as np
import gc

## Creating a dataset
Before we can fine-tune a model, we need a dataset. This dataset will contain images of paper and PMD. In the file explorer on the left, you will see a folder *dataset*; inside it you will find two subfolders: *paper* and *PMD*. These folders contain the images we are going to train on.
In the following code cells, we load the images from each folder into a list. We have already provided a number of functions that make it easier to process the data; these are in the *helpers* library. Below, we call a function that takes two parameters. The first parameter is the folder with images of PMD, the second parameter is the label for the items in that folder. The function will load the images in the folder and convert them to images of 240x240 pixels.

In [ ]:
# We laden alle afbeeldingen in de map 'dataset/pmd' in en geven ze de label 'PMD'
afbeeldingen_pmd, labels_pmd = helpers.laadt_bestanden_in_map_met_label("dataset/pmd", label="PMD")

Now that we have loaded the PMD images, we’ll take a look at what they look like. In the cell below, you can see the code to display various properties of our dataset.

In [ ]:
print(f"De dataset bevat {len(afbeeldingen_pmd)} afbeeldingen met label 'PMD'")
print(f"De labels zijn: {labels_pmd}")
print(f"De eerste afbeelding heeft een grootte van {afbeeldingen_pmd[0].shape}")
print("De eerste zes afbeeldingen zien er als volgt uit:")
helpers.toon_afbeeldingen(afbeeldingen_pmd, labels_pmd, max_afbeeldingen=6)

**Assignment:**1. Find the size of the images in the output above. **Tip:** It consists of three numbers.2. Give the meaning of each of these three numbers?

**Assignment**: Complete the code cells below so that you save the Paper images into a variable. Replace the ___ in the appropriate places with the correct code.- **Tip 1:** The code cells below are similar to the two code cells above.- **Tip 2:** We can refer to a folder in the file explorer with a *relative path*. For the PMD dataset, that was *dataset/pmd*. Note that subfolders in the path are separated by a */*.- **Tip 3:** We write the label with a capital letter!

In [ ]:
# We laden alle afbeeldingen in de map 'dataset/papier' in en geven ze de label 'Papier'
afbeeldingen_papier, labels_papier = helpers.laadt_bestanden_in_map_met_label("___", label="___")

In [ ]:
print(f"De dataset bevat {len(___)} afbeeldingen met label 'Papier'")
print(f"De labels zijn: {___}")
print(f"De eerste afbeelding heeft een grootte van {___}")
print("De eerste zes afbeeldingen zien er als volgt uit:")
helpers.toon_afbeeldingen(___, ___, max_afbeeldingen=6)

## Preparing the data for the AI system
Now that we have loaded our images and labels into Python, we can process them into a format that the AI system requires. To do so, we will go through the following steps.1. Combine the images of PMD and paper into a single dataset.2. Convert the labels from text to numbers.3. Split the dataset into three sets.* **The training set**: we use this to train our AI system.* **The validation set**: we use this to test the performance of the AI system during development. The images in this set do not overlap with the training set. This set is needed to see whether the AI system can generalize and thus has not simply memorized the images in the training set.* **The test set**: we use this to validate the performance of the AI system after development. The images in this set do not overlap with those in the train and test sets.4. Enrich the training set by rotating, rescaling, and flipping the existing images. This is called *data augmentation*.    

Do you want to know why it's necessary to have training, validation, and test sets? Then complete the activity on recognizing emotions at [dwengo.org/waisda](dwengo.org/waisda).

### Step 1: combining the images and labels
With the code below, we combine all images and labels. The result is two numpy arrays, one with the images and one with the labels.

In [ ]:
afbeeldingen = np.vstack([np.array(afbeeldingen_pmd), np.array(afbeeldingen_papier)])
labels = np.concatenate([np.array(labels_pmd), np.array(labels_papier)])

Print information about the arrays.

In [ ]:
print(f"De dataset bevat {afbeeldingen.shape[0]} afbeeldingen")
print(f"Er zijn {len(labels)} labels")
print(f"De eerste afbeelding heeft een grootte van {afbeeldingen[0].shape}")

**Assignment**:1. Check the format of the dataset.2. Does this match the sum of the number of images of PMD and paper?

### Step 2: Convert the labels from text to numbers.
Because computers can compute faster and more efficiently with numbers, we convert our labels from text to numbers. Here we use **one-hot** encoding. We will represent each label by two numbers. The first pair of numbers is the label for *PMD*. Here the first number is a 1 and the second is a 0. The second pair of numbers is the label for *Paper*. Here the first number is a 0 and the second is a 1. In the image below you can visually see how the labels for PMD and Paper look.
![](images/voorbeeld_one_hot.png)

In [ ]:
# Deze code zal onze labels one-hot encoderen.
labels_one_hot = helpers.one_hot_encode_labels(labels, ["PMD", "Papier"])

Now that we have the new labels, we can print 10 random images with their new label.

In [ ]:
# Genereer 10 willekeurige indices.
random_indices = np.random.randint(0, len(labels), 10)
# Toon deze 10 willekeurige afbeeldingen.
helpers.toon_afbeeldingen(afbeeldingen[random_indices], labels_one_hot[random_indices], max_afbeeldingen=10)

**Assignment**1. View the images above and their labels.2. Check that the label consistently matches the object you see in the image.

### Step 3: split the dataset into training, test and validation sets.
To split our dataset into these three sets, we use the *train_test_split* function from the *sklearn* library. We use it twice: first to create a validation set, then to create the test and training sets.

#### Setting up the validation set
The following code cell will randomly select 20% of the dataset as the validation set.

In [ ]:
overige_afbeeldingen, validatie_afbeeldingen, overige_labels, validatie_labels = train_test_split(afbeeldingen, labels_one_hot, test_size=0.2)

View the size of the validation set and the set with the remaining images.

In [ ]:
print(f"De overige dataset bevat {overige_afbeeldingen.shape[0]} afbeeldingen")
print(f"De validatieverzameling bevat {validatie_afbeeldingen.shape[0]} afbeeldingen")

**Assignment**: Complete the code below so that:* The *overige_afbeeldingen* and *overige_labels* are split into training set and test set.* 10% of the remaining images are used as a test set.* **Tip:** base this on the code cell above.

In [ ]:
# Vul deze code aan op de plaatsen waar ___ staat.
train_afbeeldingen, test_afbeeldingen, train_labels, test_labels = train_test_split(___, ___, test_size=___)

In [ ]:
print(f"De trainingsverzameling bevat {___} afbeeldingen")
print(f"De testverzameling bevat {___} afbeeldingen")

### Step 4: enrich the dataset (data augmentation).
Because we are working with a relatively small dataset here, it may be a good idea to artificially augment this dataset. We can do this by slightly distorting the images in our dataset. In this way, the AI model can also learn what variations of objects in the dataset look like. This should help the model generalize better and thus be better at recognizing objects it has not yet seen.

For this we use the *ImageDataGenerator* function from the Keras library. We can use it to automatically generate variations of an existing image. These variations are obtained by randomly applying the following operations to an image:* Rotate* Shift horizontally* Move vertically* Zoom in and out* Mirroring

In [ ]:
datagenerator = ImageDataGenerator(
    rotation_range=20,      # Draai de afbeelding met maximaal 20 graden
    width_shift_range=0.2,  # Verschuif de afbeelding horizontaal met maximaal 20%
    height_shift_range=0.2, # Verschuif de afbeelding verticaal met maximaal 20%
    zoom_range=0.15,        # Zoom in and uit met 15%
    horizontal_flip=True,   # Spiegel de afbeelding horizontaal
    fill_mode="nearest"     # Vul de lege pixels op met de dichtstbijzijnde pixel
)

We can apply this data generator multiple times to each image in our training set. The code below will generate 5 variations for each image in the dataset.

In [ ]:
augmented_afbeeldingen = []
augmented_labels = []

n_augmented = 5  # Aantal variaties per afbeelding

for i in range(len(train_afbeeldingen)):
    image = train_afbeeldingen[i]
    label = train_labels[i]
    
    # Voeg de originele afbeelding toe aan de augmented afbeeldingen lijst
    augmented_afbeeldingen.append(image)
    augmented_labels.append(label)
    
    # Geef de afbeelding de juiste vorm.
    image = np.expand_dims(image, axis=0)
    
    # Genereer n_augmented variaties van de afbeelding.
    aug_iter = datagenerator.flow(image, batch_size=1)
    for _ in range(n_augmented):
        aug_image = next(aug_iter)[0].astype('uint8')
        augmented_afbeeldingen.append(aug_image)
        augmented_labels.append(label)  # Gebruik hetzelfde label voor de gegenereerde afbeeldingen.

# Zet om naar numpy arrays.
augmented_afbeeldingen = np.array(augmented_afbeeldingen)
augmented_labels = np.array(augmented_labels)

**Assignment:** How many images are there in the enriched dataset? Write the code below to find out.

**Assignment:** Write code in the next cell to display 10 random images from the enriched dataset on the screen. **Tip:** Base this on the code we previously used to display random images.

**Assignment:**1. Run the code you wrote in the previous cell multiple times. Each time, view the 10 random images from the augmented dataset.2. Analyze what *data augmentation* does to the images?

## Loading ImageNet
First of all, we load the existing AI model. We will further adapt this model in the notebook.

In [ ]:
imagenet = keras.applications.MobileNetV2(
    weights="imagenet",
)

We can view the structure of the ImageNet network with the following command.

In [ ]:
print(imagenet.summary())

You can see that the model has 3 504 872 *Trainable params*, i.e., trainable parameters. These are the weights of the network that were adjusted during the training process.
In the displayed table, you can see many rows. Each row corresponds to a layer in the neural network. Using the following command, you can count how many layers the network has.

In [ ]:
print(len(imagenet.layers))

The ImageNet model can identify 1,000 different objects. The code below prints a list of these objects. You will see that the labels are in English.

In [ ]:
helpers.druk_imagenet_labels_af()

**Assignment:**1. View the list of ImageNet labels.2. Find a number of objects that belong to the PMD or paper groups.

We can use the ImageNet model to make a prediction on some of the images in our training set. The code below will do that for the first 10 images in our training set.

In [ ]:
voorspellingen = imagenet.predict(augmented_afbeeldingen[0:10])

# Zet de voorspellingen om naar labels.
voorspellingen_met_label = decode_predictions(voorspellingen, top=1)

# Druk de voorspellingen af.
for i, voorspelling in enumerate(voorspellingen_met_label):
    (id, label, score) = voorspelling[0]
    print(f"Het model zegt met {score:.4f} zekerheid dat afbeelding {i} een {label} is.")


You can see that the model doesn't really know what is in the images. Still, it is useful to reuse the model for our task. After all, the model is already able to extract certain features (e.g., lines and shapes) from the images. That is useful for our application as well.

## Expanding ImageNet
For our task, recognizing PMD and paper, we are going to extend the ImageNet model.

In [ ]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model

Below we add two new layers to the ImageNet model. A *Dense* layer with 1024 neurons and a *Dense* layer with 2 neurons.

In [ ]:
# Sla de laatste laag van het model op.
x = imagenet.output

# Voeg een nieuwe vollig verbonden laag toe met 1024 neuronen en een relu activatiefunctie.
x = Dense(1024, activation='relu')(x)

# Voeg een nieuwe laag toe die de voorspellingen doet van onze one-hot encoded labels.
predictions = Dense(2, activation='softmax')(x)  

# Combineer het bestaande model met de nieuwe lagen.
uitgebreid_model = Model(inputs=imagenet.input, outputs=predictions)

**Assignment:**1. Print the number of layers of the new model.2. Is it correct that the new model has two more layers than the ImageNet model?

In [ ]:
# Druk het aantal lagen in het uitgebreide model af.
print(len(uitgebreid_model.layers))

### Training the new layers
We now only want to train the new layers in the model. We can do this by "freezing" the layers of the original ImageNet model. We do this by setting the layers' *trainable* attribute to *False*.

In [ ]:
# Bevries de lagen van het ImageNet model.
for laag in imagenet.layers:
    laag.trainable = False

Now we can train the new layers in the custom model with the images in our training set. When you run the next cell, you will see that the network starts learning based on our training set. You will see various details.* Which *Epoch* we are in. This indicates how many times we have presented the entire training set as an example to the network.* The *accuracy* is calculated by dividing the number of correct predictions by the total number of predictions. The higher the accuracy, the better the network's performance on the training set. You will see that you can view both the *accuracy* on the training set (accuracy) and that on the validation set (val_accuracy).* The *loss* indicates how much the network's predictions deviate on average from the correct value. The higher the loss, the worse the network's performance on the training set.

In [ ]:
uitgebreid_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
uitgebreid_model.fit(augmented_afbeeldingen, augmented_labels, epochs=7, batch_size=32, validation_data=(validatie_afbeeldingen, validatie_labels))

**Assignment:**1. View the output of the above cell.2. Has the model learned how to distinguish between images of paper and PMD?3. Explain why you think the model has or has not learned anything.

### Testing the AI system
Now that we have a model that can detect PMD and paper, we can test its results on our test set.

In [ ]:
# Bereken de accuracy op de testverzameling.
test_loss, test_accuracy = uitgebreid_model.evaluate(test_afbeeldingen, test_labels)
print(f"Test accuracy: {test_accuracy}")

In [ ]:
# Doe een voorspelling voor de afbeeldingen in de testverzameling.
predictions = uitgebreid_model.predict(test_afbeeldingen)

In [ ]:
# Toon de afbeeldingen met hun voorspelling.
mapped_labels_true = ["PMD" if np.argmax(label) == 0 else "Papier" for label in test_labels]
mapped_labels_predicted = ["PMD" if np.argmax(label) == 0 else "Papier" for label in predictions]
mapped_labels_combined = [f"Echt: {mapped_labels_true[i]} \n Voorspeld: {mapped_labels_predicted[i]}" for i in range(len(mapped_labels_true))]
helpers.toon_afbeeldingen(test_afbeeldingen, mapped_labels_combined, max_afbeeldingen=len(test_afbeeldingen))

## Saving the model
Now that we have adjusted the weights of our model based on our data, we can save the model to a file. We can then download that file and import it back into another application. To save the model to a file, we use the code below.

In [ ]:
# Sla het model op in een bestand.
uitgebreid_model.save("model.h5")
# Geef het geheugen vrij
del imagenet
del uitgebreid_model  # Verwijder het model.
gc.collect()  # Geef het GPU geheugen vrij memory

In [ ]:
# Maak download links voor de bestanden.
helpers.maak_jupyterhub_download_link("model.h5", link_text="Klink hier om je model te downloaden.")

In [ ]:
# Maak download links voor de bestanden.
helpers.maak_jupyterhub_download_link("scripts/test_het_model_lokaal.py", link_text="Klink hier om het script te downloaden.")

You will see that a new file has been added on the left in the file explorer. You can download that file and use it in your own Python application. Do you want to try the model on a live video stream from the webcam? Then you can download the file *test_het_model_lokaal.py* for that, which you can find in the *scripts* folder. By placing that script, along with the saved model, in a folder on your computer and running the Python script, you can make predictions on a live video stream from your webcam.
**Assignment:**1. Download your model and the file *test_het_model_lokaal.py*.2. Place the two files in the same folder on your computer.3. Run the Python script on your own computer.    * Note that you need the libraries opencv, tensorflow, keras and numpy for that.    * The script attempts to install these automatically. To do this, you may need to run the script with administrator privileges.* **Tip:** It can be useful to run the script in a virtual Python environment. This way you don't "pollute" your global Python installation with various libraries. More info about virtual Python environments can be found at [https://docs.python.org/3/library/venv.html](https://docs.python.org/3/library/venv.html)


# Partners
This teaching material is inspired by the Smart Trash Can project by Robbe Wulgaert. If you want to get started with that project, you can find everything about it on [Robbe's website](https://www.robbewulgaert.be/onderwijs/bouw-een-slimme-vuilnisbak).
This material was developed by Dwengo vzw and was made possible with the support of VLAIO.
!["VLAIO logo"](images/vlaio.png)
!["Dwengo logo"](images/dwengo-groen-zwart.png)